# FCA-MoE v0.2 — Positive and Negative Controls (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msellamiTN/FCA-MoE/blob/main/notebooks/FCA_MoE_v02_Controls.ipynb)

Validate recovery of known overlapping expert groups in a positive control and non-recovery in an IID negative control. This is a synthetic methodology test, not a pretrained LLM benchmark. Router training and evaluation use disjoint windows; FCA is built only from held-out router predictions.

In [ ]:
from pathlib import Path
import os, sys, subprocess, platform, json
REPO=Path('/content/FCA-MoE')
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','--depth','1','https://github.com/msellamiTN/FCA-MoE.git',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
print('Repository:',REPO,'Python:',platform.python_version())

In [ ]:
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'available:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select a Colab GPU runtime.')
props=torch.cuda.get_device_properties(0); gpu_name=torch.cuda.get_device_name(0)
print('GPU:',gpu_name,'compute:',f'{props.major}.{props.minor}','VRAM GiB:',round(props.total_memory/2**30,2))
if 'T4' not in gpu_name.upper(): print('WARNING: assigned GPU is not T4; record the actual hardware.')

In [ ]:
%cd /content/FCA-MoE
!python -m pip install -q -e .
!python -m unittest discover -s tests -v

## 1. Baseline positive and negative controls

Positive: one of four overlapping four-expert communities persists inside each 64-token window. Negative: the same route-pair vocabulary is sampled IID per token, without persistent window membership. Each token still selects Top-2.

In [ ]:
from fca_moe.v02_experiment import run_controlled_case, save_json

positive=run_controlled_case(mode='window_correlated',n_samples=4096,window_size=64,activation_rate_threshold=0.10,epochs=120,seed=17)
negative=run_controlled_case(mode='iid',n_samples=4096,window_size=64,activation_rate_threshold=0.10,epochs=120,seed=17)
keys=['mode','gpu_name','torch_version','n_train_samples','n_validation_samples','route_exact_match_validation','n_validation_windows','concepts_total_unfiltered','concept_enumeration_complete','concepts_order4','expected_order4_recovered_count','expected_order4_recall','order4_precision','training_seconds','trace_seconds','fca_enumeration_seconds']
print('POSITIVE CONTROL'); print(json.dumps({k:positive.get(k) for k in keys},indent=2))
print('NEGATIVE IID CONTROL'); print(json.dumps({k:negative.get(k) for k in keys},indent=2))

assert positive['route_exact_match_validation']>=0.98
assert positive['concept_enumeration_complete'] is True
assert positive['concepts_total_unfiltered']==10 and positive['concepts_order4']==4
assert positive['expected_order4_recall']==1.0 and positive['order4_precision']==1.0
assert negative['route_exact_match_validation']>=0.98
assert negative['concept_enumeration_complete'] is True
assert negative['concepts_total_unfiltered']==1 and negative['concepts_order4']==0
assert negative['expected_order4_recall']==0.0
print('BASELINE CONTROL GATES: PASSED')

## 2. Inspect the recovered order-four intents

In [ ]:
for c in positive['concepts']:
    if len(c['intent'])==4: print('intent=',c['intent'],'extent=',c['extent_size'],'support=',round(c['support'],4))
print('Complete lattice concepts:',positive['concepts_total_unfiltered'])
print('Expected intents:',positive['expected_order4_intents'])

## 3. Sensitivity analysis

Vary window sizes 32/64/128 and thresholds 0.10/0.20/0.30. Interpret precision and unexpected order-four intents alongside recall: threshold binarization can generate spurious concepts even in IID data.

In [ ]:
from fca_moe.v02_experiment import run_sensitivity_grid
import pandas as pd
sensitivity=run_sensitivity_grid(modes=('window_correlated','iid'),window_sizes=(32,64,128),thresholds=(0.10,0.20,0.30),n_samples=4096,epochs=120,seed=17)
sensitivity_df=pd.DataFrame(sensitivity)
display(sensitivity_df)

In [ ]:
from pathlib import Path
record={'experiment_version':'0.2.0','runtime':{'gpu_name':gpu_name,'torch_version':torch.__version__,'cuda_version':torch.version.cuda,'python_version':platform.python_version(),'total_vram_bytes':props.total_memory},'positive_control':positive,'negative_iid_control':negative,'sensitivity':sensitivity}
Path('results').mkdir(exist_ok=True)
save_json(record,'results/fca_moe_v02_t4_controls.json')
sensitivity_df.to_csv('results/fca_moe_v02_sensitivity.csv',index=False)
print('Saved JSON and CSV to results/.')

## Interpretation limits

- Passing validates only the synthetic FCA/context pipeline.
- Short windows or aggressive thresholds can create spurious concepts; do not report recall without precision/false-positive counts.
- The two-device placement result is a virtual cost-model simulation, not physical multi-GPU communication.
- No pretrained LLM, Unsloth fine-tuning, expert GPU isolation or CPU/GPU weight cache is tested. Next stage is real pretrained-MoE routing-trace capture.